# S6E9 · 05：继续 04 的融合路线

**当前对照来自你的已完成结果**：04 的 `50% CatBoost + 50% LightGBM`，开发 OOF **0.94598624**，你报告的 Kaggle 公榜 **0.94618**。这不是本 notebook 的新实测成绩。

本轮不训练 CatBoost，也不重跑 04 的 LightGBM。**A 部分读取旧预测，只补一个 25% CatBoost + 75% LightGBM 的比例；B 部分可选，只训练一套新 LightGBM（深度 5→6，叶子上限仍为 31），再与旧模型固定比例融合。**

默认 `RUN_NEW_LIGHTGBM = False`，直接 Run All 只会完成旧结果核对和 A 部分，不会开始训练。进行 B 部分时，在第 4 节明确改为 `True` 后运行其后的单元格。

独立打开这个 05 即可，不需要运行 03 或 04，也不会执行它们的 notebook 单元格。需要原项目 `data` 和 04 比较目录所引用的旧模型预测目录。所有新输出保存到 `outputs/v5_*`，不覆盖旧结果，不自动提交 Kaggle。

本文件通过语法、Notebook 结构和小型合成数据流程测试；**没有在你的真实竞赛数据上运行 A/B 实验，不保证提分。**

## 1. 加载函数（不训练）

行内特征、内层五折目标编码、ID/折划分检查直接延续 04。训练目标编码不能提前用全体训练标签计算；验证与测试映射只来自对应的外层训练折。新候选沿用这些规则。

代码会从 **指定的 04 比较目录的 `continuation_summary.json`** 找到原 CatBoost 和原 LightGBM 的路径，不会随便选一个最新目录。05 模型按数据哈希、参数、版本和代码指纹复用已完成的折；无法从中断那一折的某棵树接着训练。

模型 `.txt` 不含全部特征编码管线；后续推理需保留同目录的 `feature_code.py`、折训练索引和相同原始数据，重建映射。

In [1]:
from __future__ import annotations
import gc
import hashlib
import importlib.metadata
import json
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "Will_Buy_EV"
NUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
           "Environmental_Concern_Level"]


def features(frame, model_name):
    """Row-local features only: no labels or full-data target statistics."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()
    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),
                                 ("Daily_Commute_km", 10, 3),
                                 ("Charging_Stations_Near_Home", 1, 2),
                                 ("Charging_Stations_Near_Work", 1, 2)]:
        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)
    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)
    x["income_last_2"] = income % 100
    x["income_last_3"] = income % 1000
    if model_name == "catboost":
        for col in NUMERIC:
            scale = 10 if col == "Daily_Commute_km" else 1
            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)
    else:
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000
    return x

def encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):
    """Inner OOF for training TE; outer-training-only maps for validation/test."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y)
    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))
    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]
    for col in keys:
        encoded = np.full(len(a), np.nan)
        for fit_idx, hold_idx in inner:
            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})
            stats = subset.groupby("key")["y"].agg(["sum", "count"])
            prior = labels[fit_idx].mean()
            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()
        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])
        prior = labels.mean()
        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
        a[f"{col}_te"] = encoded
        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()
        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()
        assert np.isfinite(encoded).all()
    # Label-free frequency: exclude a training row's own count contribution.
    counts = a["Annual_Income_USD"].value_counts()
    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)
    for frame in (b, c):
        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)
    for col in cats:
        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))
        for frame in (a, b, c):
            frame[col] = frame[col].astype(dtype)
    return a, b, c

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def input_hashes(project):
    root = Path(project) / "data"
    names = ["train.csv", "test.csv", "sample_submission.csv"]
    return {name: file_hash(root / name) for name in names if (root / name).is_file()}

def probabilities(values, expected_length, description):
    p = np.asarray(values, dtype=float)
    require(p.shape == (expected_length,), f"{description}: wrong prediction shape")
    require(np.isfinite(p).all(), f"{description}: NaN or infinity")
    require(((p >= 0) & (p <= 1)).all(), f"{description}: probabilities outside [0, 1]")
    return p

def align_csv(path, ids, columns):
    """Do not blend CSVs by their incidental row order: align explicitly by id."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"缺少文件：{path}。请先确认原 outputs 目录；尚未开始新训练。")
    frame = pd.read_csv(path)
    require(set(["id", *columns]).issubset(frame.columns), f"{path.name}: missing columns")
    require(frame.id.notna().all() and frame.id.is_unique, f"{path.name}: invalid/duplicate ids")
    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted) and wanted.isin(frame.id).all(), f"{path.name}: id sets differ")
    return frame.set_index("id").loc[wanted, columns].reset_index()

def load_project(project):
    project = Path(project)
    for name in ("train.csv", "test.csv"):
        if not (project / "data" / name).is_file():
            raise FileNotFoundError(f"找不到 {project / 'data' / name}；请检查 PROJECT。")
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    require(TARGET in train, "train.csv is missing the target")
    require(train.drop(columns=TARGET).columns.tolist() == test.columns.tolist(), "Train/test schema differs")
    require(train.id.is_unique and test.id.is_unique, "Duplicate id")
    require(not train.id.isin(test.id).any(), "Train/test ids overlap")
    require(not train.isna().any().any() and not test.isna().any().any(), "Missing values found")
    require(set(train[TARGET].unique()) == {"No", "Yes"}, "Expected No/Yes labels")
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    require(y.value_counts().min() >= 10, "Too few rows per class for nested five-fold validation")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    sample = project / "data/sample_submission.csv"
    if sample.exists():
        sample_frame = pd.read_csv(sample)
        require(sample_frame.columns.tolist() == ["id", TARGET], "Unexpected submission columns")
        require(sample_frame.id.is_unique and len(sample_frame) == len(test), "Invalid sample submission ids")
        require(pd.Index(test.id).isin(sample_frame.id).all(), "Sample submission has different ids")
    return train, test, y, folds

def read_run(run_dir, model_name, train, test, y, folds, expected_auc=None):
    """Verify saved OOF, test averages, and the actual five validation splits."""
    run_dir = Path(run_dir)
    config_path = run_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"找不到 {config_path}；请把 BASELINE_DIR 指向原训练结果目录。")
    config = json.loads(config_path.read_text())
    require(config.get("folds") == 5 and config.get("seed") == 42, "Run does not use V3 folds/seed")
    require(model_name in config.get("models", []), f"This run does not contain {model_name}")
    require(config.get("train_rows") == len(train) and config.get("test_rows") == len(test), "Dataset sizes differ")
    oof_frame = align_csv(run_dir / f"oof_{model_name}.csv", train.id, ["target", "prediction"])
    test_frame = align_csv(run_dir / f"submission_{model_name}.csv", test.id, [TARGET])
    require(np.array_equal(oof_frame.target.to_numpy(), y.to_numpy()), "OOF labels do not match current train.csv")
    oof = probabilities(oof_frame.prediction, len(train), "OOF")
    test_pred = probabilities(test_frame[TARGET], len(test), "Test")
    reconstructed = np.full(len(train), np.nan)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test))
    fold_records = []
    for fold, (_, vi) in enumerate(folds, start=1):
        path = run_dir / f"{model_name}_fold{fold}_predictions.npz"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；无法核对原五折，不会静默混用其他划分。")
        with np.load(path, allow_pickle=False) as saved:
            require(np.array_equal(saved["valid_indices"], vi), f"Fold {fold}: split differs")
            require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), f"Fold {fold}: train order/ids differ")
            require(np.array_equal(saved["test_ids"], test.id), f"Fold {fold}: test order/ids differ")
            vp = probabilities(saved["valid_prediction"], len(vi), f"Fold {fold} OOF")
            tp = probabilities(saved["test_prediction"], len(test), f"Fold {fold} test")
            reconstructed[vi] = vp
            coverage[vi] += 1
            test_sum += tp
            fold_records.append({"fold": fold, "auc": float(roc_auc_score(y.iloc[vi], vp))})
    require((coverage == 1).all(), "OOF coverage is not exactly once per row")
    require(np.allclose(reconstructed, oof, atol=1e-12, rtol=0), "OOF CSV differs from fold predictions")
    require(np.allclose(test_sum / 5, test_pred, atol=1e-12, rtol=0), "Test CSV is not this run's five-fold average")
    score = float(roc_auc_score(y, oof))
    if expected_auc is not None:
        require(abs(score - expected_auc) < 1e-8,
                f"旧 OOF AUC 是 {score:.10f}，不是本次预期的 {expected_auc:.10f}；请检查目录版本。")
    return {"oof": oof, "test": test_pred, "auc": score, "fold_scores": pd.DataFrame(fold_records)}

V4_FEATURE_SOURCE = 'from __future__ import annotations\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport sys\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import StratifiedKFold\n\nTARGET = "Will_Buy_EV"\nNUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",\n           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",\n           "Environmental_Concern_Level"]\n\n\ndef features(frame, model_name):\n    """Row-local features only: no labels or full-data target statistics."""\n    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()\n    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),\n                                 ("Daily_Commute_km", 10, 3),\n                                 ("Charging_Stations_Near_Home", 1, 2),\n                                 ("Charging_Stations_Near_Work", 1, 2)]:\n        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)\n        for digit in range(positions):\n            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)\n    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)\n    x["income_last_2"] = income % 100\n    x["income_last_3"] = income % 1000\n    if model_name == "catboost":\n        for col in NUMERIC:\n            scale = 10 if col == "Daily_Commute_km" else 1\n            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)\n    else:\n        x["income_prefix_100"] = income // 100\n        x["income_prefix_1000"] = income // 1000\n    return x\n\ndef encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):\n    """Inner OOF for training TE; outer-training-only maps for validation/test."""\n    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]\n    labels = np.asarray(train_y)\n    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))\n    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()\n    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]\n    for col in keys:\n        encoded = np.full(len(a), np.nan)\n        for fit_idx, hold_idx in inner:\n            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})\n            stats = subset.groupby("key")["y"].agg(["sum", "count"])\n            prior = labels[fit_idx].mean()\n            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)\n            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()\n        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])\n        prior = labels.mean()\n        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)\n        a[f"{col}_te"] = encoded\n        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()\n        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()\n        assert np.isfinite(encoded).all()\n    # Label-free frequency: exclude a training row\'s own count contribution.\n    counts = a["Annual_Income_USD"].value_counts()\n    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)\n    for frame in (b, c):\n        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)\n    for col in cats:\n        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))\n        for frame in (a, b, c):\n            frame[col] = frame[col].astype(dtype)\n    return a, b, c\n'


REFERENCE = "v4_cat50_lgb50"
# Preserve insertion order: the proven V4 blend wins an exact OOF tie.
PART_A_SPECS = {
    REFERENCE: {"catboost": 0.50, "lgb04": 0.50},
    "cat25_lgb04_75": {"catboost": 0.25, "lgb04": 0.75},
    "lgb04_only": {"lgb04": 1.00},
    "cat75_lgb04_25": {"catboost": 0.75, "lgb04": 0.25},
    "catboost_only": {"catboost": 1.00},
}
# One new model configuration, NOT a grid search.
V5_SPEC = {
    "n_estimators": 5000, "learning_rate": 0.05,
    "max_depth": 6, "num_leaves": 31,  # V4 max_depth=5; everything else unchanged.
    "min_child_samples": 100, "reg_lambda": 5.0,
    "early_stopping_rounds": 200, "te_smoothing": 1.0,
}
PART_B_NEW_SPECS = {
    "lgb05_only": {"lgb05": 1.0},
    "cat50_lgb05_50": {"catboost": 0.50, "lgb05": 0.50},
    "cat50_lgb04_25_lgb05_25": {"catboost": 0.50, "lgb04": 0.25, "lgb05": 0.25},
}

def save_json(path: Path, value: dict) -> None:
    """Atomic writes prevent incomplete metadata from being marked complete."""
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)


def package_versions() -> dict:
    return {name: importlib.metadata.version(name)
            for name in ("numpy", "pandas", "scikit-learn", "lightgbm")}


def check_source_files(bundle: dict) -> None:
    require(input_hashes(bundle["project"]) == bundle["hashes"],
            "原始 CSV 已改变，请停止并核对，不要混用不同版本的数据。")
    require(file_hash(bundle["compare_dir"] / "continuation_summary.json") == bundle["summary_sha256"],
            "04 的比较摘要已改变；请重新加载并核对来源。")


def load_v4(project: Path, compare_dir: Path,
            expected_oof: float | None = 0.94598624) -> dict:
    """Read ONLY the explicitly selected V4 run. Never guess the newest folder."""
    project, compare_dir = Path(project), Path(compare_dir)
    summary_file = compare_dir / "continuation_summary.json"
    if not summary_file.is_file():
        raise FileNotFoundError(f"找不到 {summary_file}。请核对 04 第 5 节打印的比较目录。")
    summary = json.loads(summary_file.read_text(encoding="utf-8"))
    require(summary.get("selected_by_development_oof", {}).get("candidate") == "blend_cat50_lgb50",
            "这个目录不是截图中的 04 的 50/50 最优融合，不能悄悄换成别的运行。")
    train, test, y, folds = load_project(project)
    hashes = input_hashes(project)
    require(summary.get("input_sha256") == hashes,
            "当前数据哈希与 04 记录不一致。请恢复当时的数据，不要跳过检查。")
    cat_dir = Path(summary["baseline_dir"])
    lgb_dir = Path(summary["lightgbm_dir"])
    fingerprint = lgb_dir / "data_fingerprints.json"
    if not fingerprint.is_file():
        raise FileNotFoundError(f"找不到 {fingerprint}。请保留 04 的原 LightGBM 输出目录。")
    require(json.loads(fingerprint.read_text()) == hashes, "04 的 LightGBM 数据版本不一致。")
    cat = read_run(cat_dir, "catboost", train, test, y, folds)
    lgb4 = read_run(lgb_dir, "lightgbm", train, test, y, folds)
    base = {"oof": 0.5 * cat["oof"] + 0.5 * lgb4["oof"],
            "test": 0.5 * cat["test"] + 0.5 * lgb4["test"]}
    saved_oof = align_csv(compare_dir / "oof_blend_cat50_lgb50.csv", train.id,
                          ["target", "prediction"])
    saved_test = align_csv(compare_dir / "submission_next_candidate.csv", test.id, [TARGET])
    require(np.array_equal(saved_oof.target.to_numpy(), y.to_numpy()), "04 融合 OOF 标签不一致。")
    require(np.allclose(saved_oof.prediction, base["oof"], atol=1e-12, rtol=0),
            "04 的融合 OOF 不等于两个来源模型的 50/50。")
    require(np.allclose(saved_test[TARGET], base["test"], atol=1e-12, rtol=0),
            "04 的候选提交不等于来源模型的 50/50。")
    base["auc"] = float(roc_auc_score(y, base["oof"]))
    require(abs(base["auc"] - summary["selected_by_development_oof"]["oof_auc"]) < 1e-10,
            "04 摘要分数与保存的预测不一致。")
    if expected_oof is not None:
        require(abs(base["auc"] - expected_oof) < 1e-8,
                f"载入的融合 OOF {base['auc']:.10f} 与截图的 {expected_oof:.8f} 不一致。")
    bundle = {"project": project, "compare_dir": compare_dir, "cat_dir": cat_dir,
              "lgb04_dir": lgb_dir, "hashes": hashes, "summary_sha256": file_hash(summary_file),
              "train": train, "test": test, "y": y, "folds": folds,
              "base": base, "models": {"catboost": cat, "lgb04": lgb4}}
    check_source_files(bundle)
    print("04 结果核对通过。不会重训旧 CatBoost 或旧 LightGBM。")
    print("CatBoost 来源：", cat_dir)
    print("LightGBM 来源：", lgb_dir)
    print(f"当前 04 融合 OOF：{base['auc']:.10f}")
    print("当前已报告公榜：0.94618（来自你的截图，不是本程序新跑出的成绩）")
    return bundle


def blend_predictions(models: dict, weights: dict) -> dict:
    require(bool(weights) and all(float(w) >= 0 for w in weights.values()), "融合权重必须非负。")
    require(abs(sum(weights.values()) - 1.0) < 1e-12, "融合权重之和必须为 1。")
    require(all(name in models for name in weights), "缺少所需模型预测。")
    return {part: sum(float(w) * models[name][part] for name, w in weights.items())
            for part in ("oof", "test")}


def submission_frame(bundle: dict, pred: np.ndarray) -> pd.DataFrame:
    test = bundle["test"]
    pred = probabilities(pred, len(test), "Submission")
    frame = pd.DataFrame({"id": test.id.to_numpy(), TARGET: pred})
    sample = bundle["project"] / "data/sample_submission.csv"
    if sample.is_file():
        ids = pd.read_csv(sample, usecols=["id"])
        frame = ids.merge(frame, on="id", how="left", validate="one_to_one", sort=False)
    probabilities(frame[TARGET], len(test), "Ordered submission")
    return frame


def evaluate_and_save(bundle: dict, specs: dict, prefix: str) -> tuple:
    """Compare with the CURRENT 04 fusion, not the old CatBoost-only baseline."""
    check_source_files(bundle)
    require(REFERENCE in specs, "必须保留 04 融合对照。")
    y, folds, base = bundle["y"], bundle["folds"], bundle["base"]
    reference = blend_predictions(bundle["models"], specs[REFERENCE])
    for part in ("oof", "test"):
        require(np.allclose(reference[part], base[part], atol=1e-12, rtol=0), "04 对照被意外改变。")
    base_folds = [float(roc_auc_score(y.iloc[vi], base["oof"][vi])) for _, vi in folds]
    records, per_fold = [], []
    best_name, best_score, best_prediction = REFERENCE, base["auc"], reference
    for name, weights in specs.items():
        pred = blend_predictions(bundle["models"], weights)
        probabilities(pred["oof"], len(y), f"{name} OOF")
        probabilities(pred["test"], len(bundle["test"]), f"{name} test")
        score = float(roc_auc_score(y, pred["oof"]))
        deltas = []
        for fold, (_, vi) in enumerate(folds, 1):
            fa = float(roc_auc_score(y.iloc[vi], pred["oof"][vi]))
            delta = fa - base_folds[fold - 1]
            deltas.append(delta)
            per_fold.append({"candidate": name, "fold": fold, "auc": fa,
                             "delta_vs_v4_blend": delta})
        records.append({"candidate": name, "oof_auc": score,
                        "delta_vs_v4_blend": score - base["auc"],
                        "improved_folds_vs_v4": sum(d > 0 for d in deltas),
                        "worst_fold_delta_vs_v4": min(deltas)})
        if score > best_score:
            best_name, best_score, best_prediction = name, score, pred
    check_source_files(bundle)
    out = bundle["project"] / "outputs" / (prefix + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    comparison = pd.DataFrame(records).sort_values("oof_auc", ascending=False, kind="stable").reset_index(drop=True)
    by_fold = pd.DataFrame(per_fold)
    comparison.to_csv(out / "comparison.csv", index=False)
    by_fold.to_csv(out / "comparison_by_fold.csv", index=False)
    submission_frame(bundle, best_prediction["test"]).to_csv(out / "submission_next_candidate.csv", index=False)
    pd.DataFrame({"id": bundle["train"].id.to_numpy(), "target": y.to_numpy(),
                  "prediction": best_prediction["oof"]}).to_csv(out / "oof_selected.csv", index=False)
    # Save the explicitly tested new 25/75 candidate even when it does not win.
    if "cat25_lgb04_75" in specs:
        p25 = blend_predictions(bundle["models"], specs["cat25_lgb04_75"])
        submission_frame(bundle, p25["test"]).to_csv(out / "submission_cat25_lgb04_75.csv", index=False)
    identical = bool(np.array_equal(best_prediction["test"], base["test"]))
    summary = {"selected": best_name, "selected_oof_auc": best_score,
               "selected_weights": specs[best_name], "reference_oof_auc": base["auc"],
               "user_reported_reference_public_auc": 0.94618,
               "identical_to_v4_predictions": identical, "candidates": records,
               "candidate_weights": specs, "input_sha256": bundle["hashes"],
               "v4_compare_dir": str(bundle["compare_dir"]),
               "v4_summary_sha256": bundle["summary_sha256"],
               "lgb05_dir": str(bundle.get("lgb05_dir", "")),
               "note": "Development OOF used for early stopping and candidate selection; not an independent test. No new Kaggle score known."}
    save_json(out / "summary.json", summary)
    print(comparison.to_string(index=False, float_format=lambda value: f"{value:.8f}"))
    print("\n结果目录：", out)
    print("按开发 OOF 选择：", best_name)
    print("候选 CSV：", out / "submission_next_candidate.csv")
    if identical:
        print("预测与当前 04 提交相同：保留 0.94618，不需要重复提交。")
    else:
        print("先检查逐折变化，再决定是否提交；OOF 微小提升不等于显著提升或保证公榜提升。")
    return out, comparison, by_fold, summary


def train_lgb05(bundle: dict, threads: int = -1, spec: dict | None = None) -> tuple:
    """A new five-fold LightGBM configuration. Resume COMPLETE folds only."""
    try:
        import lightgbm as lgb
    except (ImportError, OSError) as error:
        raise RuntimeError("当前内核不能导入 lightgbm。请使用已成功运行 04 的内核；缺包时用 %pip install lightgbm。") from error
    spec = dict(V5_SPEC if spec is None else spec)
    require(set(spec) == set(V5_SPEC), "训练参数键不完整，不能混用未知配置。")
    require(spec["num_leaves"] <= 2 ** spec["max_depth"], "叶子数与深度配置不一致。")
    check_source_files(bundle)
    current_versions = package_versions()
    old_versions_file = bundle["lgb04_dir"] / "package_versions.json"
    if not old_versions_file.is_file():
        raise FileNotFoundError(f"缺少 {old_versions_file}；请保留完整的 04 输出。")
    old_versions = json.loads(old_versions_file.read_text())
    require(current_versions == old_versions,
            f"当前环境版本与 04 不同，不能把差异全归因于深度。请切回运行 04 的环境。\n04: {old_versions}\n当前: {current_versions}")
    experiment = {"models": ["lightgbm"], "folds": 5, "seed": 42,
                  "train_rows": len(bundle["train"]), "test_rows": len(bundle["test"]),
                  "spec": spec, "threads": threads, "input_sha256": bundle["hashes"],
                  "package_versions": current_versions,
                  "feature_source_sha256": hashlib.sha256(V4_FEATURE_SOURCE.encode()).hexdigest(),
                  "trainer_revision": "v5-depth6-leaves31-1", "v4_summary_sha256": bundle["summary_sha256"]}
    signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()[:16]
    out = bundle["project"] / "outputs" / f"v5_lightgbm_d{spec['max_depth']}_{signature}"
    out.mkdir(parents=True, exist_ok=True)
    config_file = out / "config.json"
    if config_file.is_file():
        require(json.loads(config_file.read_text()) == experiment, "已有缓存配置不同，停止以防混用。")
    else:
        require(not any(out.iterdir()), "新目录中有未标记文件，不能安全复用。")
        save_json(config_file, experiment)
        save_json(out / "data_fingerprints.json", bundle["hashes"])
        save_json(out / "package_versions.json", current_versions)
        (out / "feature_code.py").write_text(V4_FEATURE_SOURCE, encoding="utf-8")
    print("05 LightGBM 目录：", out)
    print("仅重训新配置；已完整保存且通过核对的折会跳过。中断那一折从头开始。")
    train, test, y, folds = bundle["train"], bundle["test"], bundle["y"], bundle["folds"]
    x, xt = features(train, "lightgbm"), features(test, "lightgbm")
    require(x.columns.tolist() == xt.columns.tolist(), "训练和测试特征不一致。")
    oof, test_sum = np.full(len(train), np.nan), np.zeros(len(test))
    records = []
    for fold, (ti, vi) in enumerate(folds, 1):
        pred_file = out / f"lightgbm_fold{fold}_predictions.npz"
        model_file = out / f"lightgbm_fold{fold}.txt"
        index_file = out / f"lightgbm_fold{fold}_train_indices.npy"
        marker_file = out / f"fold{fold}_complete.json"
        if marker_file.is_file():
            marker = json.loads(marker_file.read_text())
            for filename, checksum in marker["file_sha256"].items():
                path = out / filename
                require(path.is_file() and file_hash(path) == checksum,
                        f"第 {fold} 折缓存缺失/被改动：{filename}。停止，不混用。")
            require(np.array_equal(np.load(index_file, allow_pickle=False), ti), "缓存训练折不一致。")
            with np.load(pred_file, allow_pickle=False) as saved:
                require(np.array_equal(saved["valid_indices"], vi), "缓存验证折不一致。")
                require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), "缓存验证 ID 不一致。")
                require(np.array_equal(saved["test_ids"], test.id), "缓存测试 ID 不一致。")
                vp = probabilities(saved["valid_prediction"], len(vi), "Cached OOF")
                tp = probabilities(saved["test_prediction"], len(test), "Cached test")
            best = int(marker["best_trees"])
            print(f"复用已完成 Fold {fold}/5，best_trees={best}")
        else:
            print(f"\n========== 新 LightGBM | Fold {fold}/5 ==========", flush=True)
            a, b, c = encode_fold(x.iloc[ti], y.iloc[ti], x.iloc[vi], xt,
                                  seed=1041 + fold, smoothing=spec["te_smoothing"])
            bins = [1023 if col == "Annual_Income_USD" else 255 for col in a.columns]
            model_params = {key: value for key, value in spec.items()
                            if key not in {"early_stopping_rounds", "te_smoothing"}}
            model = lgb.LGBMClassifier(
                **model_params, max_bin_by_feature=bins, objective="binary", metric="auc",
                random_state=41 + fold, n_jobs=threads, verbosity=-1,
                deterministic=True, force_col_wise=True,
            )
            model.fit(a, y.iloc[ti], eval_set=[(b, y.iloc[vi])], eval_metric="auc",
                      callbacks=[lgb.early_stopping(spec["early_stopping_rounds"], first_metric_only=True),
                                 lgb.log_evaluation(400)])
            require(np.array_equal(model.classes_, [0, 1]), "正类概率列不符合预期。")
            vp = probabilities(model.predict_proba(b)[:, 1], len(vi), "New OOF")
            tp = probabilities(model.predict_proba(c)[:, 1], len(test), "New test")
            best = int(model.best_iteration_)
            model.booster_.save_model(str(model_file))
            np.save(index_file, ti)
            with pred_file.with_suffix(".npz.tmp").open("wb") as handle:
                np.savez_compressed(handle, valid_indices=vi, valid_ids=train.id.iloc[vi].to_numpy(),
                                    valid_prediction=vp, test_ids=test.id.to_numpy(), test_prediction=tp)
            pred_file.with_suffix(".npz.tmp").replace(pred_file)
            save_json(marker_file, {"fold": fold, "best_trees": best,
                      "file_sha256": {p.name: file_hash(p) for p in (model_file, index_file, pred_file)}})
            del a, b, c, model
            gc.collect()
        oof[vi] = vp
        test_sum += tp
        score = float(roc_auc_score(y.iloc[vi], vp))
        records.append({"model": "lightgbm", "fold": fold, "auc": score, "best_trees": best})
        pd.DataFrame(records).to_csv(out / "fold_scores.csv", index=False)
        print(f"Fold AUC: {score:.10f}", flush=True)
    probabilities(oof, len(train), "Completed OOF")
    pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": oof}).to_csv(
        out / "oof_lightgbm.csv", index=False)
    pd.DataFrame({"id": test.id.to_numpy(), TARGET: test_sum / len(folds)}).to_csv(
        out / "submission_lightgbm.csv", index=False)
    check_source_files(bundle)
    result = read_run(out, "lightgbm", train, test, y, folds)
    save_json(out / "summary.json", {"oof_auc": result["auc"], "spec": spec,
              "note": "Single-model development OOF only; final blend comparison is a separate step."})
    print(f"新 LightGBM 单模型 OOF：{result['auc']:.10f}；请继续与 04 融合比较。")
    return out, result


## 2. 定位并核对你获得 0.94618 的 04 结果

下面的目录取自你截图中的实际路径。若移动过项目或删除过旧输出，请调整路径/恢复目录，而不是删除检查代码。核对通过后应打印 `当前 04 融合 OOF：0.94598624...`。

这一部分不导入 LightGBM，也不训练。

In [2]:
PROJECT = Path("/Users/yw/unsw/Kaggle")
V4_COMPARE_DIR = PROJECT / "outputs/v4_compare_20260930_163707_027699"

bundle = load_v4(PROJECT, V4_COMPARE_DIR)
print("当前 Python：", sys.executable)

04 结果核对通过。不会重训旧 CatBoost 或旧 LightGBM。
CatBoost 来源： /Users/yw/unsw/Kaggle/outputs/v3_20260925_180457_278867
LightGBM 来源： /Users/yw/unsw/Kaggle/outputs/v4_lightgbm_20260930_163526_079778
当前 04 融合 OOF：0.9459862434
当前已报告公榜：0.94618（来自你的截图，不是本程序新跑出的成绩）
当前 Python： /opt/miniconda3/envs/dl/bin/python


## 3. A：只补测 25% CatBoost + 75% 旧 LightGBM（不训练）

04 比较过 CatBoost 占比 100%、75%、50%、0%；这次只增加 **25%**，其余四行是复现对照。没有做 1% 步长的密集权重搜索。

所有 `delta_vs_v4_blend` 和 `improved_folds_vs_v4` 都相对于**当前 04 的 50/50 融合**，不是相对于旧 CatBoost。`improved_folds_vs_v4` 不是显著性检验。

若 `v4_cat50_lgb50` 仍然最好，保留当前提交，不重复提交。新比例即使更高，仍需检查逐折变化，不保证公榜/私榜更高。

In [3]:
part_a_dir, part_a_comparison, part_a_by_fold, part_a_summary = evaluate_and_save(
    bundle, PART_A_SPECS, "v5A_blend_"
)
part_a_comparison

     candidate    oof_auc  delta_vs_v4_blend  improved_folds_vs_v4  worst_fold_delta_vs_v4
cat25_lgb04_75 0.94602102         0.00003477                     5              0.00000863
v4_cat50_lgb50 0.94598624         0.00000000                     0              0.00000000
    lgb04_only 0.94594638        -0.00003986                     1             -0.00009085
cat75_lgb04_25 0.94585286        -0.00013338                     0             -0.00015591
 catboost_only 0.94561823        -0.00036801                     0             -0.00041436

结果目录： /Users/yw/unsw/Kaggle/outputs/v5A_blend_20260930_170339_414823
按开发 OOF 选择： cat25_lgb04_75
候选 CSV： /Users/yw/unsw/Kaggle/outputs/v5A_blend_20260930_170339_414823/submission_next_candidate.csv
先检查逐折变化，再决定是否提交；OOF 微小提升不等于显著提升或保证公榜提升。


,candidate,oof_auc,delta_vs_v4_blend,improved_folds_vs_v4,worst_fold_delta_vs_v4
0,cat25_lgb04_75,0.946021,0.000035,5,0.000009
1,v4_cat50_lgb50,0.945986,0.000000,0,0.000000
2,lgb04_only,0.945946,-0.000040,1,-0.000091
3,cat75_lgb04_25,0.945853,-0.000133,0,-0.000156
4,catboost_only,0.945618,-0.000368,0,-0.000414


In [4]:
# 每一列都是该方案相对于当前 04 融合的逐折 AUC 变化。
part_a_by_fold.pivot(index="fold", columns="candidate", values="delta_vs_v4_blend")

candidate,cat25_lgb04_75,cat75_lgb04_25,catboost_only,lgb04_only,v4_cat50_lgb50
fold,,,,,
1,0.000036,-0.000131,-0.000356,-0.000036,0.0
2,0.000009,-0.000114,-0.000338,-0.000091,0.0
3,0.000018,-0.000112,-0.000321,-0.000072,0.0
4,0.000057,-0.000156,-0.000414,-0.000003,0.0
5,0.000055,-0.000156,-0.000413,0.000006,0.0


## 4. B：可选，仅训练一套新的 LightGBM

A 部分不需要训练；**准备做模型实验时才将下面 `False` 改成 `True`**。

| 设置 | 04 LightGBM | 05 新候选 |
|---|---:|---:|
| max_depth | 5 | **6** |
| num_leaves | 31 | 31 |
| n_estimators | 5000 | 5000 |
| learning_rate | 0.05 | 0.05 |
| min_child_samples | 100 | 100 |
| reg_lambda | 5.0 | 5.0 |
| 早停等待 | 200 | 200 |
| 目标编码 smoothing | 1.0 | 1.0 |

只放宽深度约束，保持叶子数上限不变；不是保证新模型更强。保持原特征、外层/内层折划分和每折种子。程序会核对依赖版本；请用已成功运行 04 的同一 Python 环境。

`THREADS = -1` 与原设置一致。电脑同时做其他工作时可改小，但会被记为一个不同的缓存配置。

**不会重训原来的任何模型。** 完整保存的 05 折会复用；没有完整完成标记的折从头训练。

In [7]:
RUN_NEW_LIGHTGBM = True  # 开始新 LightGBM 实验时才改成 True。
THREADS = -1

lgb05_dir = None
if RUN_NEW_LIGHTGBM:
    lgb05_dir, lgb05_result = train_lgb05(bundle, threads=THREADS)
    bundle["models"]["lgb05"] = lgb05_result
    bundle["lgb05_dir"] = lgb05_dir
else:
    # 防止同一内核此前的 B 结果被误当成本次已启动实验。
    bundle["models"].pop("lgb05", None)
    bundle.pop("lgb05_dir", None)
    print("B 部分未开始训练。A 部分结果已经可用。")

05 LightGBM 目录： /Users/yw/unsw/Kaggle/outputs/v5_lightgbm_d6_80604546b1a3083a
仅重训新配置；已完整保存且通过核对的折会跳过。中断那一折从头开始。

========== 新 LightGBM | Fold 1/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.944895
Early stopping, best iteration is:
[499]	valid_0's auc: 0.944902
Evaluated only: auc
Fold AUC: 0.9449024595

========== 新 LightGBM | Fold 2/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.945554
Early stopping, best iteration is:
[406]	valid_0's auc: 0.945559
Evaluated only: auc
Fold AUC: 0.9455592114

========== 新 LightGBM | Fold 3/5 ==========


...

Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.94675
Early stopping, best iteration is:
[414]	valid_0's auc: 0.946753
Evaluated only: auc
Fold AUC: 0.9467530864

========== 新 LightGBM | Fold 4/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.946224
Early stopping, best iteration is:
[480]	valid_0's auc: 0.946247
Evaluated only: auc
Fold AUC: 0.9462466009

========== 新 LightGBM | Fold 5/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.945944
Early stopping, best iteration is:
[513]	valid_0's auc: 0.945961
Evaluated only: auc
Fold AUC: 0.9459609426
新 LightGBM 单模型 OOF：0.9458783458；请继续与 04 融合比较。


## 5. B：加入新 LightGBM 后，固定比例比较

仅在第 4 节训练完成后运行。保留 A 的全部候选，再增加三项：新 LightGBM 单模型、`50% CatBoost + 50% 新 LightGBM`、`50% CatBoost + 25% 旧 LightGBM + 25% 新 LightGBM`。

最后一项等于保留 CatBoost 总权重，并把 LightGBM 部分分给两种配置。**这是预先规定的待验证方案，不是已证实最优的配置。** 本轮不把新模型与每个新权重做全组合搜索。

新模型单独分数高/低都不能替代融合评估；看最终组合相对于 04 的 OOF 与逐折变化。若 A 的新比例已经更好，也一同保留在选择池中。

`submission_next_candidate.csv` 只是开发 OOF 选出的下一份候选，不代表应当无条件覆盖当前公榜最佳提交。

In [8]:
if RUN_NEW_LIGHTGBM and "lgb05" in bundle["models"]:
    part_b_specs = {**PART_A_SPECS, **PART_B_NEW_SPECS}
    part_b_dir, part_b_comparison, part_b_by_fold, part_b_summary = evaluate_and_save(
        bundle, part_b_specs, "v5B_compare_"
    )
    display(part_b_comparison)
    display(part_b_by_fold.pivot(index="fold", columns="candidate", values="delta_vs_v4_blend"))
else:
    print("没有启动新 LightGBM，本节跳过。先查看第 3 节 A 的结果。")

              candidate    oof_auc  delta_vs_v4_blend  improved_folds_vs_v4  worst_fold_delta_vs_v4
         cat25_lgb04_75 0.94602102         0.00003477                     5              0.00000863
         v4_cat50_lgb50 0.94598624         0.00000000                     0              0.00000000
cat50_lgb04_25_lgb05_25 0.94598383        -0.00000241                     2             -0.00000636
         cat50_lgb05_50 0.94596349        -0.00002275                     0             -0.00002969
             lgb04_only 0.94594638        -0.00003986                     1             -0.00009085
             lgb05_only 0.94587835        -0.00010790                     0             -0.00014601
         cat75_lgb04_25 0.94585286        -0.00013338                     0             -0.00015591
          catboost_only 0.94561823        -0.00036801                     0             -0.00041436

结果目录： /Users/yw/unsw/Kaggle/outputs/v5B_compare_20260930_171919_096073
按开发 OOF 选择： cat25_lgb04_75
候

,candidate,oof_auc,delta_vs_v4_blend,improved_folds_vs_v4,worst_fold_delta_vs_v4
0,cat25_lgb04_75,0.946021,0.000035,5,0.000009
1,v4_cat50_lgb50,0.945986,0.000000,0,0.000000
2,cat50_lgb04_25_lgb05_25,0.945984,-0.000002,2,-0.000006
3,cat50_lgb05_50,0.945963,-0.000023,0,-0.000030
4,lgb04_only,0.945946,-0.000040,1,-0.000091
5,lgb05_only,0.945878,-0.000108,0,-0.000146
6,cat75_lgb04_25,0.945853,-0.000133,0,-0.000156
7,catboost_only,0.945618,-0.000368,0,-0.000414


candidate,cat25_lgb04_75,cat50_lgb04_25_lgb05_25,cat50_lgb05_50,cat75_lgb04_25,catboost_only,lgb04_only,lgb05_only,v4_cat50_lgb50
fold,,,,,,,,
1,0.000036,-0.000006,-0.000030,-0.000131,-0.000356,-0.000036,-0.000134,0.0
2,0.000009,0.000002,-0.000016,-0.000114,-0.000338,-0.000091,-0.000122,0.0
3,0.000018,-0.000006,-0.000026,-0.000112,-0.000321,-0.000072,-0.000146,0.0
4,0.000057,0.000001,-0.000017,-0.000156,-0.000414,-0.000003,-0.000060,0.0
5,0.000055,-0.000004,-0.000025,-0.000156,-0.000413,0.000006,-0.000078,0.0


## 如何使用输出

`v5A_blend_*` 和 `v5B_compare_*` 下的 `comparison.csv` / `comparison_by_fold.csv` 是比较记录；`summary.json` 保存权重、来源、数据指纹和所选方案；`oof_selected.csv` 是选中方案的开发 OOF；`submission_next_candidate.csv` 是相应测试集概率。只有最后这个是下一份候选提交文件。

A 另外保存 `submission_cat25_lgb04_75.csv` 供追溯，不代表即使它输了也要提交。若摘要 `identical_to_v4_predictions=true`，预测与当前 04 相同，不必重复提交。

**对照不要混淆**：本地对比使用融合 OOF **0.94598624**；实际新提交评分再与已报告公榜 **0.94618** 比较。OOF 参与了早停和候选选择，不是独立评估；多折同向也不等于统计显著性。不要依据公榜无限增加权重或参数候选。

算法来源：本会话的 `04_reuse_catboost_lightgbm.ipynb`，`features`、`encode_fold` 及旧结果核对函数保持原实现；新功能为当前融合对照、有限候选、单一 LightGBM 参数实验及完整折复用。

技术参考：[LightGBM 参数控制](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html)、[目标编码内部交叉拟合](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_target_encoder_cross_val.html)、[模型选择与评估偏差](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)。